# EDA básico — preliminar
Análisis descriptivo de una versión curada. Sustituya `VERSION` por la ejecución que desea revisar.
No contiene datos ni resultados precalculados. Las decisiones de selección o transformación
basadas en los datos deben realizarse dentro de entrenamiento; el EDA no acredita capacidad predictiva.


In [ ]:
from pathlib import Path

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'src' / 'ingesta.py').is_file() and (p / 'config' / 'data_schema.yml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde la raíz del repositorio o notebooks/.')

import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt

DATA = ROOT / 'data' / 'processed' / 'VERSION' / 'model_table.csv'
if not DATA.is_file():
    raise FileNotFoundError('Indique una versión real generada por src/preprocesamiento.py.')
report = json.loads((DATA.parent / 'data_quality_report.json').read_text(encoding='utf-8'))
if hashlib.sha256(DATA.read_bytes()).hexdigest() != report['output_sha256']:
    raise ValueError('La tabla cambió respecto de su reporte de curaduría.')
roles = report['roles']
df = pd.read_csv(
    DATA,
    dtype={c: 'string' for c in roles['identifiers'] + roles['groups']},
    keep_default_na=False, na_values=[''],
)
numeric_columns = roles['predictors'] + [roles['target']]
df.head()


In [ ]:
df.info()
df.describe(include='all').T


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
df[numeric_columns].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()


## Revisión antes del modelamiento
Verificar unidades, target, exclusiones, duplicados, constantes, redundancia y anomalías físicas
contra el reporte y la fuente. Confirmar réplicas y grupos antes de diseñar las particiones.
Los identificadores, grupos y `__source_record` son trazabilidad, nunca predictores.
